<img src="images/logo.jpg" style="display: block; margin-left: auto; margin-right: auto;" alt="לוגו הקורס">

# <p style="text-align: right; direction: rtl; float: right;">SQL ופייתון: יבוא CSV ושמירת תוצאות</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
במחברת הזו נחבר בין מה שלמדנו ביום 3 (קבצי CSV ו-JSON) לבין מסדי נתונים:<br>
🔹 ניקח קובץ CSV של עובדים ונטען אותו למסד נתונים<br>
🔹 נשתמש ב-<code>with</code> כדי לשמור שינויים בצורה בטוחה<br>
🔹 נקבל תוצאות כמילונים (<code>sqlite3.Row</code>) במקום tuples<br>
🔹 נשמור את תוצאות השאילתות לקבצי JSON ו-CSV
</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/tip.png" style="height: 50px !important;" alt="טיפ">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>חשוב להבין</strong>:<br>
            אם קיבלתם שגיאה באמצע (למשל <code>Cannot operate on a closed database</code>) -
הריצו מחדש מהתא <strong>"התחברות למסד הנתונים"</strong> ומטה. כל התאים בנויים להרצה חוזרת.
        </p>
    </div>
</div>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">התחברות למסד הנתונים</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
נתחבר למסד נתונים חדש בשם <code>hr.db</code> ("משאבי אנוש").<br>
אם הרצתם את התא כבר פעם, הוא סוגר את החיבור הישן קודם.
</p>

In [ ]:
import sqlite3
import csv
import json
import os

try:
    conn.close()
except NameError:
    pass  # הרצה ראשונה - אין חיבור ישן

conn = sqlite3.connect('hr.db')
print("מחוברים ל-hr.db ✅")

## <p style="text-align: right; direction: rtl; float: right; clear: both;">הפקודה with - שמירה בטוחה של שינויים</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
עד עכשיו כתבנו <code>conn.commit()</code> אחרי כל שינוי. אבל מה קורה אם באמצע הפעולה משהו משתבש?<br>
עם <code>with conn:</code> פייתון עושה את זה בשבילנו:<br>
🔹 אם כל הפקודות בתוך ה-<code>with</code> הצליחו - מתבצע <code>commit</code> אוטומטי<br>
🔹 אם הייתה שגיאה - מתבצע <code>rollback</code> אוטומטי, וכל מה שנעשה בתוך הבלוק מתבטל<br><br>
שימו לב: <code>with conn</code> <strong>לא סוגר</strong> את החיבור, רק מטפל ב-commit / rollback.<br>
נשתמש גם בקיצור <code>conn.execute(...)</code>, שיוצר cursor בשבילנו.<br><br>
נתחיל ביצירת טבלת עובדים. העמודות זהות לאלה שבקובץ ה-CSV:
</p>

In [ ]:
with conn:
    conn.execute('DROP TABLE IF EXISTS staff')
    conn.execute('''
        CREATE TABLE staff (
            id INTEGER PRIMARY KEY,
            first_name TEXT NOT NULL,
            last_name TEXT NOT NULL,
            email TEXT,
            department TEXT,
            salary INTEGER,
            hire_date TEXT
        )
    ''')

print("הטבלה staff נוצרה ✅")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
עכשיו נראה את ה-rollback האוטומטי בפעולה. ננסה להכניס שתי שורות עם אותו <code>id</code> - אסור, כי הוא PRIMARY KEY (ייחודי).<br>
השורה השנייה תיכשל, ואנחנו מצפים שגם הראשונה תתבטל:
</p>

In [ ]:
try:
    with conn:
        conn.execute("INSERT INTO staff VALUES (1, 'דני', 'כהן', 'dani@example.com', 'הנדסה', 15000, '2020-03-15')")
        conn.execute("INSERT INTO staff VALUES (1, 'כפילות', 'בכוונה', 'x@example.com', 'הנדסה', 1, '2020-01-01')")
except sqlite3.IntegrityError as error:
    print("❌ קיבלנו שגיאה:", error)

count = conn.execute('SELECT COUNT(*) FROM staff').fetchone()[0]
print("מספר שורות בטבלה:", count)

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 השורה הראשונה נכנסה בהצלחה, אבל השנייה גרמה ל-<code>IntegrityError</code><br>
🔹 בגלל ה-<code>with</code>, כל הבלוק בוטל - גם השורה הראשונה לא נשמרה, והטבלה נשארה ריקה (0 שורות)<br>
🔹 זה בדיוק מה שרוצים: או שהכל מצליח, או שדבר לא משתנה. לפעולות כאלה קוראים <strong>טרנזקציה</strong>
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">יבוא קובץ CSV למסד הנתונים</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
בתיקייה <code>resources</code> יש קובץ <code>employees.csv</code> עם 10 עובדים. נטען אותו לטבלה בשלושה שלבים:<br>
1. קוראים את הקובץ עם <code>csv.DictReader</code> (כמו ביום 3)<br>
2. הופכים כל שורה ל-tuple בסדר העמודות של הטבלה. חשוב להמיר את השכר למספר עם <code>int()</code> - בקובץ CSV הכל טקסט!<br>
3. מכניסים את כל הרשימה בבת אחת עם <code>executemany</code>
</p>

In [ ]:
# שלב 1+2: קריאת הקובץ והכנת הרשימה
rows = []
with open('../../resources/employees.csv', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for record in reader:
        rows.append((
            int(record['id']),
            record['first_name'],
            record['last_name'],
            record['email'],
            record['department'],
            int(record['salary']),      # המרה למספר
            record['hire_date'],
        ))

print(f"נקראו {len(rows)} שורות מהקובץ")
print("השורה הראשונה:", rows[0])

In [ ]:
# שלב 3: טעינה למסד הנתונים
with conn:
    conn.execute('DELETE FROM staff')   # מנקים, כדי שהרצה חוזרת לא תכפיל שורות
    conn.executemany('INSERT INTO staff VALUES (?, ?, ?, ?, ?, ?, ?)', rows)

count = conn.execute('SELECT COUNT(*) FROM staff').fetchone()[0]
print(f"בטבלה staff יש עכשיו {count} עובדים ✅")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 <code>DictReader</code> נתן לנו כל שורה כמילון, וכך בנינו tuple לפי שמות העמודות ולא לפי מספרים<br>
🔹 שבע סימני שאלה, כמספר העמודות; <code>executemany</code> הריץ את ה-INSERT פעם אחת לכל tuple ברשימה<br>
🔹 ה-<code>DELETE</code> בתחילת הבלוק גורם לכך שאפשר להריץ את התא שוב ושוב בלי לקבל כפילויות<br>
🔹 כל זה בתוך <code>with conn</code>, ולכן אם הייתה שגיאה באמצע לא היינו נשארים עם חצי טבלה
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">קבלת תוצאות כמילונים - sqlite3.Row</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
כשמבקשים שורה מהמסד מקבלים tuple, וצריך לזכור מה מספר העמודה: <code>row[5]</code> זה... שכר? תאריך? קשה לדעת.<br>
אם נגדיר <code>conn.row_factory = sqlite3.Row</code>, כל שורה תתנהג כמו מילון - אפשר לפנות אליה לפי <strong>שם העמודה</strong>.
</p>

In [ ]:
# לפני: tuple רגיל
row = conn.execute('SELECT * FROM staff WHERE id = ?', (1,)).fetchone()
print("tuple רגיל:", row)
print("השכר הוא:", row[5], "(צריך לזכור שזו עמודה מספר 5...)")

# מגדירים שהשורות יתנהגו כמו מילונים
conn.row_factory = sqlite3.Row

row = conn.execute('SELECT * FROM staff WHERE id = ?', (1,)).fetchone()
print()
print("עם sqlite3.Row:")
print("שם:", row['first_name'], row['last_name'])
print("שכר:", row['salary'])
print("שמות העמודות:", row.keys())
print("כמילון רגיל:", dict(row))

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 <code>row['salary']</code> קריא הרבה יותר מ-<code>row[5]</code><br>
🔹 <code>row.keys()</code> מראה אילו עמודות יש<br>
🔹 <code>dict(row)</code> הופך את השורה למילון פייתון רגיל - וזה בדיוק מה שנצטרך כדי לשמור ל-JSON
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">מתוצאות שאילתה לרשימה של מילונים</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
בדרך כלל נרצה להריץ שאילתה ולקבל <strong>רשימה של מילונים</strong>, כמו שמקבלים מקובץ JSON או מ-<code>csv.DictReader</code>.<br>
נכתוב פונקציה קטנה שעושה את זה פעם אחת, ונשתמש בה הלאה:
</p>

In [ ]:
def query_to_dicts(sql, params=()):
    """מריצה שאילתה ומחזירה רשימה של מילונים (שורה = מילון)"""
    cursor = conn.execute(sql, params)
    return [dict(row) for row in cursor.fetchall()]


engineers = query_to_dicts(
    'SELECT first_name, last_name, salary FROM staff WHERE department = ? ORDER BY salary DESC',
    ('הנדסה',)
)

print(f"נמצאו {len(engineers)} עובדי הנדסה:")
for person in engineers:
    print(f"- {person['first_name']} {person['last_name']}: ₪{person['salary']:,}")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 <code>[dict(row) for row in ...]</code> היא list comprehension שהופכת כל שורה למילון<br>
🔹 התוצאה היא רשימה של מילונים, בדיוק כמו ש-<code>json.dump</code> ו-<code>csv.DictWriter</code> אוהבים
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">שמירת התוצאות ל-JSON ול-CSV</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
עכשיו נסגור מעגל: CSV ← מסד נתונים ← שאילתה ← JSON ו-CSV חדשים.<br>
נשמור את העובדים עם שכר של סכום מסוים ומעלה, בתיקייה בשם <code>reports</code>.<br>
(לקובץ ה-CSV נוסיף <code>utf-8-sig</code> כדי שאקסל יציג עברית נכון, כמו שלמדנו ביום 3.)
</p>

In [ ]:
os.makedirs('reports', exist_ok=True)

min_salary = 14000  # שנו כאן
high_earners = query_to_dicts(
    'SELECT first_name, last_name, department, salary FROM staff WHERE salary >= ? ORDER BY salary DESC',
    (min_salary,)
)

# שמירה ל-JSON
with open('reports/high_earners.json', 'w', encoding='utf-8') as f:
    json.dump(high_earners, f, ensure_ascii=False, indent=2)

# שמירה ל-CSV
if high_earners:
    with open('reports/high_earners.csv', 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(high_earners[0].keys()))
        writer.writeheader()
        writer.writerows(high_earners)

print(f"נשמרו {len(high_earners)} עובדים עם שכר של ₪{min_salary:,} ומעלה")
print("הקבצים בתיקייה reports:", os.listdir('reports'))

<p style="text-align: right; direction: rtl; float: right; clear: both;">
נקרא בחזרה את קובץ ה-JSON כדי לוודא שהוא נראה כמו שצריך:
</p>

In [ ]:
with open('reports/high_earners.json', encoding='utf-8') as f:
    loaded = json.load(f)

print(json.dumps(loaded[:2], ensure_ascii=False, indent=2))

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 <code>json.dump(..., ensure_ascii=False, indent=2)</code> שמר את הרשימה לקובץ קריא, עם עברית תקינה<br>
🔹 <code>csv.DictWriter</code> כתב את אותה רשימה כקובץ CSV; <code>fieldnames</code> הוא סדר העמודות<br>
🔹 את שני הקבצים אפשר לפתוח בעורך טקסט או באקסל, או להעביר לאנשים אחרים
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">דוגמה מסכמת: דוח לפי מחלקה</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
נשלב הכל: שאילתת <code>GROUP BY</code> שמסכמת את העובדים לפי מחלקה, והתוצאה נשמרת לקובץ JSON.<br>
ככה בונים דוחות אוטומטיים: הנתונים נשארים במסד, והדוח נוצר בכל פעם מחדש.
</p>

In [ ]:
report = query_to_dicts('''
    SELECT department,
           COUNT(*) AS num_employees,
           ROUND(AVG(salary)) AS avg_salary,
           MAX(salary) AS max_salary
    FROM staff
    GROUP BY department
    ORDER BY avg_salary DESC
''')

with open('reports/department_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("דוח לפי מחלקה:")
for item in report:
    print(f"{item['department']} - מספר עובדים: {item['num_employees']}, ממוצע ₪{item['avg_salary']:,.0f}, מקסימום ₪{item['max_salary']:,}")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
🔹 כל שורה בתוצאה כבר מילון עם שמות העמודות שנתנו ב-<code>AS</code><br>
🔹 כך אפשר להפוך שאילתה לדוח JSON בכמה שורות בלבד
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">נסו בעצמכם</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            שמרו לקובץ <code>reports/hired_2020.json</code> את כל העובדים שהתקבלו לעבודה בשנת 2020.<br>
רמז: <code>hire_date</code> נשמר כטקסט בצורה <code>2020-03-15</code>, ולכן אפשר להשתמש ב-<code>WHERE hire_date LIKE '2020%'</code>.
        </p>
    </div>
</div>

In [ ]:
# נסו כאן:

## <p style="text-align: right; direction: rtl; float: right; clear: both;">סגירת החיבור</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
זה התא האחרון במחברת. אחריו אי אפשר להריץ שאילתות על <code>conn</code>.<br>
כדי להתנסות שוב - מריצים מתא "התחברות למסד הנתונים".
</p>

In [ ]:
conn.close()
print("החיבור נסגר 👋")

## <p style="text-align: right; direction: rtl; float: right; clear: both;">סיכום</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
במחברת זו למדנו:<br>
✅ <code>with conn:</code> - commit אוטומטי כשהכל הצליח, ו-rollback כשיש שגיאה<br>
✅ יבוא קובץ CSV למסד נתונים עם <code>csv.DictReader</code> ו-<code>executemany</code><br>
✅ <code>conn.row_factory = sqlite3.Row</code> - גישה לעמודות לפי שם<br>
✅ הפיכת תוצאות שאילתה לרשימה של מילונים<br>
✅ שמירת תוצאות ל-JSON ול-CSV<br><br>
עכשיו אתם מוכנים לתרגולים! 💪
</p>